# 00 Environment — Case creation, import, and review

Select one YAML file below, then run from top to bottom. The YAML controls `create`, `load`, or `edit` and selects `synthetic`, `alos`, `gsi`, or `file`. Algorithms remain in the Python packages; this notebook only orchestrates and displays results.

In [ ]:
CONFIG_FILE = "workspace/experiments/terrain_synthetic.yaml"

In [ ]:
from math import radians
from pathlib import Path
import os
import matplotlib.pyplot as plt
from field_environment import (
    CaseMode, find_repository_root, load_case, plot_environment_features,
    plot_terrain_quality, run_environment_workflow, save_case_previews,
)
from field_campost import (
    FixedCameraPlacement, create_fixed_camera_layout, heightmap_to_mesh,
    load_case_camera_layout, plot_camera_layout_2d, read_environment_usd,
    save_case_scene, validate_mesh_against_heightmap,
)

REPOSITORY_ROOT = find_repository_root()
CASES_ROOT = Path(os.environ.get("WILDLIFE_CASES_ROOT", REPOSITORY_ROOT / "cases"))

In [ ]:
workflow = run_environment_workflow(
    CONFIG_FILE, repository_start=REPOSITORY_ROOT, cases_root=CASES_ROOT
)
config = workflow.config
environment_case = workflow.environment_case
print(config.model_dump_json(indent=2))
print(f"Case directory: {workflow.case_directory}")

In [ ]:
quality_figure, _ = plot_terrain_quality(environment_case)
plt.show()
feature_figure, _ = plot_environment_features(environment_case)
plt.show()
preview_paths = save_case_previews(environment_case, workflow.case_directory)
print(*preview_paths, sep="\n")

In [ ]:
placements = tuple(
    FixedCameraPlacement(
        camera_id=item.camera_id, x_m=item.x_m, y_m=item.y_m,
        height_above_ground_m=item.height_above_ground_m,
        orientation=item.orientation(),
        horizontal_fov_rad=radians(item.horizontal_fov_deg),
        vertical_fov_rad=radians(item.vertical_fov_deg),
        max_observation_distance_m=item.max_observation_distance_m,
        observation_interval_s=item.observation_interval_s,
    )
    for item in config.cameras
)
requested_layout = create_fixed_camera_layout(
    environment_case.heightmap, placements, layout_id=f"{config.case_id}-fixed"
)
layout_path = workflow.case_directory / "camera_layout.json"
if workflow.action is CaseMode.LOAD:
    camera_layout = (load_case_camera_layout(CASES_ROOT, config.case_id)
                     if layout_path.is_file() else requested_layout)
else:
    if workflow.action is CaseMode.EDIT and layout_path.is_file() and not config.cameras:
        camera_layout = load_case_camera_layout(CASES_ROOT, config.case_id)
    else:
        camera_layout = requested_layout
    save_case_scene(
        CASES_ROOT, config.case_id, camera_layout,
        overwrite=workflow.action is CaseMode.EDIT,
    )
camera_figure, _ = plot_camera_layout_2d(
    environment_case.heightmap, environment_case.farm_boundary, camera_layout
)
plt.show()

In [ ]:
mesh = heightmap_to_mesh(environment_case.heightmap)
validate_mesh_against_heightmap(mesh, environment_case.heightmap)
reloaded_case = load_case(CASES_ROOT, config.case_id)
assert reloaded_case.case_spec.case_id == environment_case.case_spec.case_id
assert reloaded_case.heightmap.grid == environment_case.heightmap.grid
usd_path = workflow.case_directory / "environment.usd"
if usd_path.is_file():
    usd_summary = read_environment_usd(usd_path)
    assert usd_summary.layout == camera_layout
print(f"Mesh: {len(mesh.vertices)} vertices / {len(mesh.faces)} faces")
print("Case round trip: OK")